### Hybrid Search Langchain

In [ ]:
!pip install  pinecone-client pinecone-text pinecone-notebooks

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

# Load Pinecone API key from .env file (never hardcode secrets)
api_key = os.getenv("PINECONE_API_KEY")

In [ ]:
from langchain_community.retrievers import PineconeHybridSearchRetriever


In [ ]:
import os 
from pinecone import Pinecone,ServerlessSpec
index_name="hybrid-search-langchain-pinecone"
# Initialize Pinecone client 
pc=Pinecone(api_key=api_key) 

# Create Pinecone index

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384, # dimension of dense vector embeddings
        metric='dotproduct', # sparse values supported metrics 
        spec=ServerlessSpec(cloud='aws',region='us-east-1')
    )

In [ ]:
index=pc.Index(index_name)
index

In [ ]:
## vector embedding 
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["HF_TOKEN"]=os.getenv("HF_TOKEN")

from langchain_huggingface import HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
embeddings

In [ ]:
from pinecone_text.sparse import BM25Encoder

bm25_encoder=BM25Encoder().default()
bm25_encoder

In [ ]:
sentences=[
    "In 2023, I visited Paris",
        "In 2022, I visited New York",
        "In 2021, I visited New Orleans",

]

## tfidf values on these sentence
bm25_encoder.fit(sentences)

## store the values to a json file
bm25_encoder.dump("bm25_values.json")

# load to your BM25Encoder object
bm25_encoder = BM25Encoder().load("bm25_values.json")

In [ ]:
retriever=PineconeHybridSearchRetriever(embeddings=embeddings,sparse_encoder=bm25_encoder,index=index,top_k=3)

In [ ]:
retriever.add_texts(
    [
    "In 2023, I visited Paris",
        "In 2022, I visited New York",
        "In 2021, I visited New Orleans",

]
)

In [ ]:
retriever.invoke("What city did i visit first")